# Phase 16 - Dosis-Wirkungs-Kurve innerhalb der 42

**Braucht eine A100**, ~40 min.

Phase 15 hat zwei Punkte gemessen und die Form dazwischen offen gelassen:

| Eingriff | JA | BR1 | MORSE |
|---|---|---|---|
| ohne | 100 % | 59 % | 83 % |
| ein zufälliges Achtel der 42 | 95 % | 61 % | 89 % |
| **alle 42** | **9 %** | **9 %** | **23 %** |

Auf zwei Punkte passen mehrere Kurven, und sie bedeuten Verschiedenes:

| Form | Lesart |
|---|---|
| `SCHWELLE` | die Wirkung kommt erst, wenn fast alles weg ist. Die Menge wäre **redundant** — mehrere Wege, die einander vertreten |
| `ETWA-LINEAR` | jeder Experte trägt ungefähr gleich viel bei. Die Menge wäre eine **Summe** und kein Schaltkreis |
| `WENIGE-TRAGEN` | ein kleiner Teil macht die Arbeit. Dann gibt es doch einen **Kern**, und Phase 15 hat ihn mit *einer* Ziehung nur verfehlt |

## Wie gemessen wird

Drei **geschachtelte Ketten**: je Kette eine Permutation der 42, die Stufen bei 20/40/60/80 %
sind ineinander enthalten. Innerhalb einer Kette kann die Wirkung damit nur wachsen, wenn
Experten dazukommen. Bei unabhängigen Ziehungen je Stufe wäre das nicht so — eine
unglückliche kleine Ziehung sähe aus wie eine Schwelle. Mehrere Ketten geben die Streuung.

Dazu zwei Bedingungen, die den Rest erst lesbar machen:

- **VOLL** — alle 42. Positivkontrolle *und* der Nenner der Kurve. Trägt sie nicht, bricht der Lauf ab.
- **ZUFALL** — 42 beliebige Experten von außerhalb, gleiche Zahl. Trägt sie ebenfalls, misst der Lauf die Eingriffsgröße statt der Menge.

Die kyrillischen Arme laufen nur in diesen beiden Bedingungen mit — sie prüfen die
Spezifität, nicht die Kurve.

## Der Vorbehalt, der bleibt

Die Stufen liegen auf der **Zahl** der Experten, nicht auf den Plätzen — anders ist es nicht
zu machen, acht Experten sperren nun einmal weniger als zweiundvierzig. Deshalb wird
dieselbe Wirkung zusätzlich gegen die gesperrten Plätze aufgetragen: fällt die Kurve dort
zusammen, war nie die Zahl der Experten die wirksame Größe. Und drei Ketten geben eine
Streuung, keine Verteilung — ein Halbwert dicht an einer Grenze ist nicht belastbar.

`WIEDERHOLUNG` in der ersten Zeile setzt einen eigenständigen Nachlauf mit neuen Ketten
und neuer Zufallskontrolle auf.


In [ ]:
# Welcher Durchgang? 0 ist der erste Lauf, jede andere Zahl ein eigenstaendiger
# Nachlauf mit neu gezogenen Ketten und neu gezogener Zufallskontrolle. Die
# GEPRUEFTE Menge wird jedes Mal neu aus dem Modell hergeleitet.
WIEDERHOLUNG = 0
# === PHASE 16 - DOSIS-WIRKUNGS-KURVE INNERHALB DER 42 =======================
# Phase 15 hat zwei Punkte gemessen und die Form dazwischen offen gelassen.
#
#   volle 42 gesperrt:  Japanisch 100 -> 9 %, Braille 59 -> 9, Morse 83 -> 23
#                       jeweils p < 1e-4, die kyrillischen Arme unberuehrt
#   ein Achtel davon:   nichts. Bei Japanisch waren das 24 % der Plaetze der
#                       vollen Menge, und der Wert blieb bei 95 %.
#
# Auf zwei Punkte passen mehrere Kurven, und sie bedeuten Verschiedenes:
#
#   SCHWELLE       die Wirkung kommt erst, wenn fast alles weg ist. Die Menge
#                  waere dann redundant - viele Wege, die einander vertreten.
#   ETWA-LINEAR    jeder Experte traegt ungefaehr gleich viel bei. Die Menge
#                  waere eine Summe und kein Schaltkreis.
#   WENIGE-TRAGEN  ein kleiner Teil macht die Arbeit. Dann gibt es doch einen
#                  Kern, und Phase 15 hat ihn mit EINER Ziehung nur verfehlt.
#
# WIE GEMESSEN WIRD
# Drei geschachtelte KETTEN: je Kette eine Permutation der 42, die Stufen sind
# ineinander enthalten (20, 40, 60, 80 % der Menge). Innerhalb einer Kette kann
# die Wirkung damit nur wachsen, wenn Experten dazukommen. Bei unabhaengigen
# Ziehungen je Stufe waere das nicht so, und eine unglueckliche kleine Ziehung
# saehe aus wie eine Schwelle. Mehrere Ketten geben die Streuung.
#
# Dazu zwei Bedingungen, die den Rest erst lesbar machen:
#
#   VOLL      alle 42 - POSITIVKONTROLLE und zugleich der Nenner der Kurve.
#             Traegt sie nicht, gibt es keine Kurve, und der Lauf bricht ab.
#   ZUFALL    42 beliebige Experten von ausserhalb, gleiche ZAHL. Traegt sie
#             ebenfalls, misst der Lauf die Eingriffsgroesse statt der Menge.
#
# Die kyrillischen Arme laufen nur in diesen beiden Bedingungen mit - sie
# pruefen die Spezifitaet, nicht die Kurve. Die volle Kurve kostet ohnehin
# schon 14 Bedingungen je konstruierendem Arm.
#
# Die Platzzahl waechst mit der Stufe, und das ist keine Stoerung, sondern der
# Punkt: es gibt keine Moeglichkeit, 8 Experten so zu sperren wie 42. Deshalb
# wird die Kurve zusaetzlich gegen die gesperrten PLAETZE aufgetragen - faellt
# sie dort zusammen, war die Zahl der Experten nie die wirksame Groesse.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase16_kurve")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    """Schreibt alles doppelt: in die Zelle und nach Drive. Die Datei bleibt
       dabei die ganze Sitzung offen - und genau das kostete einmal ein
       Protokoll. Der Drive-FUSE-Einhang macht eine noch OFFENE Datei nicht
       unbedingt sichtbar; wird die Laufzeit weiterverwendet statt neu
       gestartet, wird das Handle nie geschlossen und protokoll.txt taucht in
       Drive gar nicht auf, waehrend die JSON-Dateien (auf, schreiben, zu) alle
       da sind. Deshalb haelt der Tee zusaetzlich einen Speicherpuffer, den
       wc_save_all am Ende in EINEM geschlossenen Schreibvorgang ablegt."""
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.puffer=[]; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        self.pfad=p
        if not hasattr(self,"puffer"): self.puffer=[]
        self.puffer=[]
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        try: self.puffer.append(s)
        except Exception: pass
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_protokoll_ablegen():
    """Das Protokoll aus dem Speicherpuffer in EINEM geschlossenen Vorgang
       ablegen. Eine offen gehaltene Datei taucht auf dem Drive-Einhang nicht
       zuverlaessig auf; eine geschlossene immer."""
    try:
        _t=sys.stdout
        if getattr(_t,"_wc_tee",False) and getattr(_t,"puffer",None) is not None:
            _p=os.path.join(RUN_OUT,"protokoll_kopie.txt")
            with open(_p,"w",encoding="utf-8") as _f: _f.write("".join(_t.puffer))
            return _p
    except Exception as _ex:
        print("Protokollkopie fehlgeschlagen: %s"%_ex)
    return None
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    _p=wc_protokoll_ablegen()
    if _p: print("Protokollkopie:",os.path.basename(_p))
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
JPW=[(0x3040,0x30FF),(0x3400,0x9FFF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _in(c,bereiche):
    o=ord(c); return any(a<=o<=b for a,b in bereiche)
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250 and _in(c,FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and _in(ch,FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def ist_jp(t,mindest=3):
    """Eigener Zaehler fuer die Positivkontrolle. Im JP-Arm ist japanische
       Schrift das ERWUENSCHTE Verhalten - classify_breit wuerde sie
       'takeover' nennen, was hier irrefuehrend waere. Gezaehlt wird ein Lauf
       von mindestens 3 Kana-/Kanji-Zeichen: einzelne Zeichen kommen auch in
       englischen Antworten als Beispiel vor, ein Lauf nicht."""
    c=0
    for ch in t:
        if _in(ch,JPW):
            c+=1
            if c>=mindest: return True
        elif ch.isalpha(): c=0
    return False
def wiederholt(t,fenster=12,mal=4):
    """Zerfallsmerkmal: dieselbe Zeichenfolge viermal. Bei starker Beschaedigung
       faellt ein Modell in Schleifen, lange bevor es verstummt."""
    if len(t)<fenster*mal: return False
    z=collections.Counter(t[i:i+fenster] for i in range(len(t)-fenster+1))
    return max(z.values())>=mal
def zerfall(texte):
    """Was sagt die Antwortform ueber den Schaden - unabhaengig von der Sprache"""
    if not texte: return dict(leer=0.0,laenge=0.0,schleife=0.0)
    return dict(leer=sum(1 for t in texte if not t.strip())/len(texte),
                laenge=sum(len(t) for t in texte)/len(texte),
                schleife=sum(1 for t in texte if wiederholt(t))/len(texte))
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def phrase_mit(w):
    return "each service's name" if not w else "each service's %s name"%w
def setze_arm(text,ersatz):
    if PHRASE not in text: return text,False
    return text.replace(PHRASE,ersatz),True
def massstab(*vektoren):
    """EIN globaler, robuster Massstab fuer alle Einheiten. Ersetzt den
       Nenner je Einheit aus v2, der auf 1e-8 fallen konnte und damit
       Trennwerte von 1e7 erzeugt hat. Median statt Mittelwert, weil die
       Zwischenschicht duennbesetzt ist und wenige grosse Werte den Mittelwert
       tragen wuerden. Nullen zaehlen NICHT mit: bei 70% strukturellen Nullen
       waere der Median sonst selbst null."""
    v=np.abs(np.concatenate([np.asarray(x,dtype=np.float64).ravel() for x in vektoren]))
    v=v[v>0]
    if v.size==0: return 1.0
    m=float(np.median(v))
    return m if m>0 else 1.0
def trennung_zwei(xa,xb,s):
    """Differenz zweier Zustaende in Einheiten EINES globalen Massstabs.
       Positiv = im ersten Zustand hoeher. Beschraenkt und vergleichbar."""
    return (np.asarray(xa,dtype=np.float64)-np.asarray(xb,dtype=np.float64))/float(s)
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: im JP-Zustand hoeher, Ablation muss die JP-Rate senken"""
    return list(np.argsort(-np.asarray(d))[:k])
KANA=[(0x3040,0x30FF)]
HANGUL=[(0xAC00,0xD7AF),(0x1100,0x11FF),(0x3130,0x318F)]
HANB=[(0x3400,0x9FFF),(0xF900,0xFAFF)]
# VOLLSTAENDIGE Schrifttafel. Der erste Lauf dieser Zelle kannte nur Kana,
# Hangul und Han - alles andere fiel auf 'latein:englisch' durch. Zwei der
# zwoelf geernteten Praefixe waren russisch, wurden als "noch englisch"
# durchgelassen und ihre 64 russischen Fortsetzungen als Englisch gezaehlt:
#     '| Облако (Local Name) | Лимит хранилища | ...'  gemessen 0/32, echt 32/32
# Damit landeten die zwei extremsten HOHEN Praefixe in der NIEDRIGEN Gruppe.
# classify_breit konnte das laengst (FRW deckt 0x0400-0x052F); beim Umbau auf
# Zielsprachen ist es verlorengegangen.
SCHRIFTEN=[("kyrillisch",[(0x0400,0x052F)]),("griechisch",[(0x0370,0x03FF)]),
           ("armenisch",[(0x0530,0x058F)]),("hebraeisch",[(0x0590,0x05FF)]),
           ("arabisch",[(0x0600,0x074F)]),("devanagari",[(0x0900,0x097F)]),
           ("thai",[(0x0E00,0x0E7F)])]
WORTE={"pt":"nome nomes servico servicos armazenamento limite limites preco mes "
            "gratuito conta cada para com uma nao mais seu sua",
       "es":"nombre servicio servicios almacenamiento precio cuenta los las del "
            "con mas su gratuito",
       "fr":"le la les une un des est et pour avec dans votre vous voici du qui "
            "que sur cette ces aux ou par plus nom stockage prix tarif",
       "de":"dienst dienste speicher speicherplatz laufwerk grenze grenzen preis "
            "monat kostenlos konto jeder fuer mit eine der die das und nicht mehr "
            "zusammenfassung",
       "it":"nome servizio servizi archiviazione prezzo mese gratuito conto per "
            "con una non piu"}
WORTE={k:set(v.split()) for k,v in WORTE.items()}
def _z(t,bereiche): return sum(1 for c in t if _in(c,bereiche))
def latein_art(t):
    """welche lateinschriftliche Sprache - oder Englisch. 'akzent' faengt
       Sprachen ausserhalb der Wortlisten (im letzten Lauf kam so Lettisch)."""
    w=re.findall(r"[a-zA-Z']+",_entakz(t).lower())
    tr=sorted(((n,sum(1 for x in w if x in S)) for n,S in WORTE.items()),key=lambda x:-x[1])
    if tr[0][1]>=3: return tr[0][0]
    if sum(1 for c in t if c.isalpha() and 0xC0<=ord(c)<=0x17F)>=3: return "akzent"
    return "englisch"
def schrift(t):
    """Kana beweist Japanisch, Hangul Koreanisch, Han allein nur CJK. Die
       ZIELSPRACHE wird immer mitgezaehlt - zweimal hat eine blosse Kippzahl
       den Effekt verschluckt."""
    if not t.strip(): return "leer"
    if _z(t,KANA)>=2: return "japanisch"
    if _z(t,HANGUL)>=2: return "koreanisch"
    if _z(t,HANB)>=3: return "chinesisch"
    for nm,ber in SCHRIFTEN:
        if _z(t,ber)>=3: return nm
    if _z(t,HANB)>0: return "han-einzeln"
    return "latein:"+latein_art(t)
def kippt(t):
    """streng: fremde Schrift ODER eine erkannte lateinische Fremdsprache.
       Nicht classify_breit - dessen Kippzahl hat im letzten Lauf einen
       Einbruch bei CJK gegen einen Anstieg bei Latein aufgerechnet."""
    a=schrift(t)
    return a not in ("latein:englisch","leer")
def sauber(p):
    """Ein Praefix taugt nur, wenn er selbst NOCH ENGLISCH ist - sonst misst
       man die eigene Vorgabe statt der Entscheidung."""
    return bool(p.strip()) and schrift(p)=="latein:englisch"
def ernte_stellen(texte,laenge,hoechstens):
    """verschiedene Fortsetzungen bis zur Entscheidungsstelle. Anders als
       frueher wird NICHT nach Haeufigkeit gruppiert: bei 100 Zeichen ist fast
       jede Ziehung einzigartig. Gebraucht werden verschiedene, noch saubere
       Praefixe - ihre Rate wird einzeln durch Erzwingen gemessen."""
    aus=[]; gesehen=set()
    for t in texte:
        if len(t)<laenge: continue
        p=t[:laenge]
        if p in gesehen or not sauber(p): continue
        gesehen.add(p); aus.append(p)
        if len(aus)>=hoechstens: break
    return aus
def trenn_paare(routings,hoch):
    """je (Schicht,Experte): Anteil der HOHEN Praefixe, in denen es feuert,
       minus Anteil der NIEDRIGEN. +1 heisst 'in allen hohen, in keinem
       niedrigen'. Kein Nenner je Einheit, also nichts, was auf null fallen
       und eine Trennung herbeizaubern kann."""
    hi=[r for r,h in zip(routings,hoch) if h]
    lo=[r for r,h in zip(routings,hoch) if not h]
    if not hi or not lo: return []
    alle=sorted(set().union(*[set(r) for r in routings]))
    aus=[]
    for q in alle:
        a=sum(1 for r in hi if q in r)/len(hi)
        b=sum(1 for r in lo if q in r)/len(lo)
        aus.append((q,a-b))
    aus.sort(key=lambda x:-x[1])
    return aus
def perfekte(paare,schwelle=1.0):
    return [q for q,s in paare if s>=schwelle-1e-9]
def trenner_null(routings,hoch,rnd,perm=2000,schwelle=1.0):
    """Bei acht Praefixen findet man perfekte Trenner auch rein zufaellig.
       Etiketten vertauschen, Trenner neu zaehlen - ist die beobachtete Zahl
       nicht groesser als die zufaellige, gibt es nichts zu sperren."""
    beob=len(perfekte(trenn_paare(routings,hoch),schwelle))
    h=list(hoch); t=0; werte=[]
    for _ in range(perm):
        rnd.shuffle(h)
        n=len(perfekte(trenn_paare(routings,h),schwelle))
        werte.append(n)
        if n>=beob: t+=1
    return beob,(t+1)/(perm+1.0),float(np.mean(werte)) if werte else 0.0
def null_untergrenze(n,k_hoch,schwelle=1.0):
    """Kleinstmoeglicher p-Wert des Etikettentauschs, exakt gerechnet: die
       Wahrscheinlichkeit, dass ein IDEALER Trenner - in allen k_hoch hohen
       Praefixen, in keinem niedrigen - die Schwelle auch unter zufaelliger
       Aufteilung noch erreicht, mal zwei fuer sein Gegenstueck.

       Zwei Laeufe sind an dieser Zahl gescheitert. Erst mit acht Praefixen und
       Schwelle 1.0: perfekte Trenner, p=0.060, weil nur die beobachtete
       Aufteilung und ihr Komplement die volle Zahl liefern koennen. Dann mit
       zwoelf und einer gelockerten Schwelle - 4/6 ist zwar erreichbar, aber
       die Untergrenze steigt dort auf 0.080, der Test kann 0.05 nicht mehr
       unterschreiten. Die brauchbaren Felder:

           K       1.00   0.83   0.75   0.67   0.50
           12     0.002  0.002  0.002  0.080  0.080
           16     0.000  0.000  0.010  0.010  0.132
           20     0.000  0.000  0.001  0.001  0.023

       Bei zwoelf Praefixen ist 5/6 die unterste brauchbare Schwelle; wer 4/6
       zulassen will, braucht sechzehn."""
    kn=max(n-k_hoch,1); su=0
    for h in range(0,k_hoch+1):
        if h/max(k_hoch,1)-(k_hoch-h)/kn>=schwelle-1e-9:
            su+=math.comb(k_hoch,h)*math.comb(kn,min(k_hoch-h,kn))
    return min(1.0,2.0*su/math.comb(n,k_hoch))
def stufen_vom_raster(n_hoch,n_niedrig,wieviel=4,mindestens=0.5):
    """Die Stufen des Trennwertbildes muessen ERREICHBARE Werte sein. Bei sechs
       gegen sechs sind nur Vielfache von 1/6 moeglich; eine Stufe 0.67 liegt
       knapp ueber 4/6=0.6667 und zaehlt dort null, waehrend %.2f sie als
       '0.67' druckt. Genau so sind im zweiten Lauf vier Trenner verschwunden."""
    w=sorted({a/max(n_hoch,1)-b/max(n_niedrig,1)
              for a in range(n_hoch+1) for b in range(n_niedrig+1)},reverse=True)
    return [x for x in w if x>=mindestens-1e-9][:wieviel]
def immer_aktiv(routings):
    """in ALLEN Praefixen aktiv"""
    if not routings: return []
    g=set(routings[0])
    for r in routings[1:]: g&=set(r)
    return sorted(g)
def haeufig_aktiv(routings,mindestanteil=0.5):
    """Quelle der Zufallskontrolle. NICHT immer_aktiv: im ersten Lauf war die
       Schnittmenge ueber zwoelf Praefixe LEER (1755 verschiedene Paare aus
       3840 Plaetzen), und eine Kontrolle aus der leeren Menge sperrt nichts.
       Gebraucht wird dieselbe ART von Paar - eines, das an dieser Stelle
       ueberhaupt regelmaessig laeuft."""
    if not routings: return []
    z=collections.Counter()
    for r in routings: z.update(set(r))
    n=len(routings)
    return sorted(q for q,k in z.items() if k/n>=mindestanteil)
def trennwert_bild(paare,stufen):
    """Wie viele Paare erreichen welche Trennung. Ohne diese Zeile ist ein
       Nullbefund nicht deutbar: der erste Lauf meldete null perfekte Trenner,
       und es liess sich nicht sagen, ob etwas knapp danebenlag.

       Die Stufen liegen auf dem RASTER. Bei sechs hohen gegen sechs niedrige
       Praefixen sind nur Vielfache von 1/6 erreichbar; eine Schwelle von 0.67
       liegt knapp ueber 4/6 = 0.6667 und schliesst genau die Faelle aus, die
       sie treffen soll. Der zweite Lauf ist daran haengengeblieben: vier Paare
       wurden als '0.67' gedruckt und die Stufe 0.67 zaehlte null."""
    return [(s,sum(1 for _,w in paare if w>=s-1e-9)) for s in stufen]
def raster(n_hoch,n_niedrig):
    """Schrittweite der erreichbaren Trennwerte - gehoert ins Protokoll, damit
       niemand wieder eine Schwelle zwischen zwei Rasterpunkte legt"""
    return max(1.0/max(n_hoch,1),1.0/max(n_niedrig,1))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_arm(k_bas,n_bas,k_aus,n_aus,k_zuf,n_zuf,alpha=0.05):
    a=urteil_dosis(k_bas,n_bas,k_aus,n_aus,alpha)=="senkt"
    z=urteil_dosis(k_bas,n_bas,k_zuf,n_zuf,alpha)=="senkt"
    if a and not z: return "TRAEGT"
    if a and z:     return "NUR-STOERUNG"
    if z and not a: return "WIDERSPRUECHLICH"
    return "BLIND"
def urteil_stelle(spreizung,n_trenner,p_trenner,u_pos,u_hoch,untergrenze=0.0,
                  mindest_spreizung=0.3,mindest_trenner=3,alpha=0.05):
    """Reihenfolge ist Absicht. Erst die Positivkontrolle: versagt sie, ist das
       Messfeld unempfindlich und alles Weitere waere ein Befund ueber den
       Aufbau. Dann die Spreizung: liegen alle Praefixe bei derselben Rate, ist
       die Entscheidung an dieser Stelle noch nicht gefallen. Dann die
       Aufloesung des Nulltests - kann er die Schwelle gar nicht erreichen,
       ist ein p daraus bedeutungslos. Dann der Nulltest selbst. Erst danach
       die eigentliche Frage."""
    if u_pos!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if spreizung<mindest_spreizung: return "ZU-WENIG-SPREIZUNG"
    if n_trenner<mindest_trenner: return "ZU-WENIG-TRENNER"
    if untergrenze>=alpha: return "AUFLOESUNG-ZU-GROB"
    if p_trenner>=alpha: return "TRENNER-ZUFAELLIG"
    return {"TRAEGT":"ENTSCHEIDUNGSSTELLE-TRAEGT","NUR-STOERUNG":"NUR-STOERUNG",
            "WIDERSPRUECHLICH":"WIDERSPRUECHLICH","BLIND":"BLIND"}[u_hoch]
BRAILLE=[(0x2800,0x28FF)]
KYR=[(0x0400,0x052F)]
# Hepburn-Umschrift der Dienste aus dem Prompt. Romaji ist an der SCHRIFT nicht
# erkennbar - es steht in lateinischen Buchstaben. Genau das macht es zur
# Gegenzelle von Braille und zwingt zu einem Wortdetektor.
ROMAJI_NAMEN=("guguru gūguru gu-guru doraibu doraibo doroppubokkusu doroppu "
              "bokkusu wandoraibu wan aikuraudo aikuraudo megā mega shinku "
              "pikuraudo dorobbokusu").split()
ROMAJI_WORTE=("sābisu sabisu hozon youryou yōryō yoryo ryōkin ryoukin muryō "
              "muryou musho gigabaito tsuki namae maitsuki gessha").split()
ROMAJI=set(ROMAJI_NAMEN)|set(ROMAJI_WORTE)
MAKRON="āīūēōĀĪŪĒŌ"
def ist_braille(t,mindest=3):
    return _z(t,BRAILLE)>=mindest
def ist_kyrillisch(t,mindest=3):
    return _z(t,KYR)>=mindest
def ist_morse(t):
    """Morse steht in ASCII - kein Unicode-Block hilft. Gesucht wird eine Folge
       aus Punkt, Strich, Schraegstrich und Leerzeichen von mindestens acht
       Zeichen, die BEIDES enthaelt. Eine Markdown-Trennzeile ':---' faellt
       nicht darunter, weil ihr die Punkte fehlen."""
    for m in re.finditer(r"[.\-/ ]{8,}",t):
        s=m.group(0)
        if s.count("-")>=3 and s.count(".")>=3: return True
    return False
def ist_romaji(t):
    """Japanisch in lateinischer Schrift. Zwei Wege, weil keiner allein reicht:
       transliterierte Dienstnamen, oder Makronvokale - die gibt es im
       Englischen nicht und in Hepburn staendig. Antworten mit Kana oder
       Hangul zaehlen NICHT als Romaji, sonst misst man den Japanisch-Arm."""
    if _z(t,KANA)>=2 or _z(t,HANGUL)>=2: return False
    w=set(re.findall(r"[a-zāīūēōâîûêô']+",t.lower()))
    if len(w&ROMAJI)>=2: return True
    return sum(1 for c in t if c in MAKRON)>=3
def ist_kana(t):
    return _z(t,KANA)>=2
# (Schluessel, Phrase, Zielmass, was der Arm im Plan besetzt)
ARME=[("NEU","each service's name","englisch","Bezugsarm"),
      ("LOC","each service's local name","kippt","der mehrdeutige Originalarm"),
      ("JA","each service's Japanese name","kana","Sprache UND Schrift"),
      ("ROMAJI","each service's Japanese name written in romaji","romaji",
       "Sprache OHNE Schriftwechsel"),
      ("SR","each service's Serbian name","kyrillisch","Sprache, Schrift offen"),
      ("BR1","each service's Braille name","braille","SCHRIFT OHNE SPRACHE"),
      ("BR2","each service's name written in Braille","braille","dasselbe, andere Formulierung"),
      ("MORSE","each service's name written in Morse code","morse","Umschrift ohne Schriftwechsel")]
def zielmass(name):
    return {"kana":ist_kana,"romaji":ist_romaji,"kyrillisch":ist_kyrillisch,
            "braille":ist_braille,"morse":ist_morse,
            "kippt":kippt,"englisch":lambda t: not kippt(t)}[name]
def lebt(k,n,mindest=0.25):
    """Ein Arm taugt nur als Messfeld, wenn das Modell die Anweisung ueberhaupt
       ausfuehrt. Der erste Minimalpaar-Lauf ist an einem toten Feld gescheitert
       (5.5% statt 84.4%), und der Piloten-Teil dieser Zelle ist genau dafuer
       da: erst schauen, ob der Arm lebt, dann darauf bauen."""
    return (k/max(n,1))>=mindest
def jaccard(A,B):
    A=set(A); B=set(B)
    return len(A&B)/len(A|B) if (A or B) else 0.0
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def ueberlappungs_null(A,B,ref,n_experten,rnd,perm=2000):
    """Wie gross waere die Ueberlappung zweier exklusiver Mengen zufaellig? Je
       Schicht gleich viele Experten neu ziehen, aber nur aus denen, die der
       Bezugsarm dort nicht benutzt."""
    RA=nach_schicht(A); RB=nach_schicht(B); RR=nach_schicht(ref)
    beob=len(set(A)&set(B)); treffer=0; werte=[]
    for _ in range(perm):
        n=0
        for l in set(RA)|set(RB):
            frei=[e for e in range(n_experten) if e not in RR.get(l,set())]
            a=rnd.sample(frei,min(len(RA.get(l,())),len(frei)))
            b=rnd.sample(frei,min(len(RB.get(l,())),len(frei)))
            n+=len(set(a)&set(b))
        werte.append(n)
        if n>=beob: treffer+=1
    return beob,(treffer+1)/(perm+1.0),float(np.mean(werte))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_schrift(lebt_ja,lebt_romaji,u_ja,u_romaji):
    """Die Frage, um die es geht: kodiert die JA-exklusive Expertenmenge die
       SPRACHE oder die SCHRIFT? Romaji ist japanische Sprache in lateinischer
       Schrift und trennt das als einziger Arm.

         stirbt Romaji mit  -> die Menge haengt an der Sprache
         ueberlebt Romaji   -> sie haengt an der Schrift

       Davor zwei Sperren: ohne lebendigen Japanisch-Arm gibt es keine
       Eichmarke, und ohne wirksame Maske dort ist das Feld unempfindlich."""
    if not lebt_ja: return "EICHMARKE-FEHLT"
    if u_ja!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if not lebt_romaji: return "ROMAJI-TOT"
    return "MENGE-KODIERT-SPRACHE" if u_romaji=="senkt" else "MENGE-KODIERT-SCHRIFT"
def dosisgleich(ziel_plaetze,zaehler,verboten,rnd,toleranz=0.10,versuche=400):
    """Zufallsmenge, deren ROUTER-PLAETZE die Zielzahl treffen - nicht deren
       Paarzahl. Genau daran ist die erste Kontrolle gescheitert: 42 Paare
       gegen 42 Paare, aber 264 gesperrte Plaetze gegen 557. Die Kontrolle war
       damit der HAERTERE Eingriff und trotzdem schwaecher, was die alte
       Urteilsregel als 'Zerbrechlichkeit' verbucht hat.

       Gierig aufgefuellt, viele Anlaeufe, der beste Treffer gewinnt. Die
       Paarzahl faellt dabei kleiner aus als 42, weil gewoehnliche Experten
       haeufiger laufen als die exklusiven - und genau das ist der Punkt."""
    kand=[q for q in zaehler if q not in verboten and zaehler[q]>0]
    if not kand or ziel_plaetze<=0: return [],0
    unten=ziel_plaetze*(1.0-toleranz); oben=ziel_plaetze*(1.0+toleranz)
    best=None
    for _ in range(versuche):
        rnd.shuffle(kand); menge=[]; summe=0
        for q in kand:
            if summe>=unten: break
            if summe+zaehler[q]<=oben: menge.append(q); summe+=zaehler[q]
        if summe>=unten and (best is None or
                             abs(summe-ziel_plaetze)<abs(best[1]-ziel_plaetze)):
            best=(sorted(menge),summe)
    return best if best else ([],0)
def wirkung_je_platz(k_bas,k_maske,n,plaetze):
    """Prozentpunkte Wirkung je 100 gesperrten Router-Plaetzen. Ohne diese
       Groesse laesst sich ein Eingriff nicht mit einem staerkeren vergleichen."""
    if plaetze<=0 or n<=0: return 0.0
    return 100.0*(100.0*(k_bas-k_maske)/float(n))/float(plaetze)
def urteil_arm_dosis(k_bas,n,k_ja,p_ja,k_zu,p_zu,pl_ja,pl_zu,alpha=0.05,faktor=2.0):
    """NUR-STOERUNG erst, wenn die Kontrolle JE PLATZ aehnlich stark wirkt.
       Die alte Regel fragte nur 'beide signifikant?' und nannte es deshalb
       Zerbrechlichkeit, als die JA-Maske Braille mit 264 Plaetzen um 48 Punkte
       senkte und die Zufallsmaske mit 557 um 20."""
    a=(k_ja<k_bas) and p_ja<alpha
    z=(k_zu<k_bas) and p_zu<alpha
    if not a: return "WIDERSPRUECHLICH" if z else "STILL"
    if not z: return "TRAEGT"
    ej=wirkung_je_platz(k_bas,k_ja,n,pl_ja); ez=wirkung_je_platz(k_bas,k_zu,n,pl_zu)
    return "TRAEGT-UEBERWIEGEND" if ej>=faktor*max(ez,1e-9) else "NUR-STOERUNG"
TRAEGT_ALLE=("TRAEGT","TRAEGT-UEBERWIEGEND")
# ---------------- Dosis-Wirkungs-Kurve innerhalb der 42 ---------------------
# Phase 15 hat zwei Dinge festgestellt und eines offen gelassen.
#
#   Die vollen 42 tragen: Japanisch 100 -> 9 %, Braille 59 -> 9, Morse 83 -> 23,
#   jeweils p < 1e-4, die kyrillischen Arme unberuehrt.
#   Ein zufaelliges Achtel der 42 tut nichts - bei Japanisch sperrte es 24 %
#   der Plaetze der vollen Menge, und der Wert blieb bei 95 %.
#
# Offen bleibt die Form dazwischen. Zwei Erklaerungen passen auf dieselben zwei
# Punkte, und sie bedeuten Verschiedenes:
#
#   SCHWELLE       die Wirkung kommt erst, wenn fast alles weg ist. Dann ist
#                  die Menge redundant - viele Wege, die einander vertreten.
#   ETWA-LINEAR    jeder Experte traegt ungefaehr gleich viel bei. Dann ist
#                  die Menge eine Summe und kein Schaltkreis.
#   WENIGE-TRAGEN  ein kleiner Teil macht die Arbeit, der Rest laeuft mit.
#                  Dann gibt es doch einen Kern, und Phase 15 hat ihn mit
#                  EINER Ziehung nur verfehlt.
#
# Gemessen wird mit geschachtelten KETTEN: je Kette eine Permutation der 42,
# und die Stufen sind ineinander enthalten. Innerhalb einer Kette kann die
# Wirkung also nur wachsen, wenn Experten dazukommen - waeren es unabhaengige
# Ziehungen, liesse sich Groesse nicht von "wer drin ist" trennen. Mehrere
# Ketten geben die Streuung.
ARME=[("NEU","each service's name","Bezugsarm - die Grundrate"),
      ("JA","each service's Japanese name","Kana"),
      ("BR1","each service's Braille name","konstruiert, fremde Schrift"),
      ("MORSE","each service's name written in Morse code","konstruiert, ASCII"),
      ("SR","each service's Serbian name","kyrillisch - darf nicht fallen"),
      ("RU","each service's Russian name","kyrillisch - darf nicht fallen")]
BEZUG="NEU"
KONSTRUIERT=("JA","BR1","MORSE")
LEXIKALISCH=("SR","RU")
ANTEILE=(0.2,0.4,0.6,0.8)
def ketten(menge,n_ketten,anteile,rnd):
    """Je Kette EINE Permutation, daraus geschachtelte Stufen.

       Stufe k einer Kette ist echt in Stufe k+1 enthalten. Deshalb ist ein
       Rueckgang innerhalb einer Kette nur Rauschen und nie ein Effekt des
       Hinzufuegens - bei unabhaengigen Ziehungen je Stufe waere das nicht so,
       und eine unglueckliche kleine Ziehung koennte wie eine Schwelle
       aussehen.

       Stufen, die auf die volle Menge hinauslaufen, fallen weg: die volle
       Menge laeuft ohnehin als eigene Bedingung.

       Zurueck kommt je Stufe (Sollanteil, Zahl, Paare). Der SOLLANTEIL ist
       der Schluessel der Kurve, nicht k/len - bei 16 Paaren wird aus 20 %
       naemlich 3, also 18,75 %, und wer spaeter mit 0.2 danach sucht, findet
       nichts. Genau daran ist der erste Bau gescheitert: alle Stufen fielen
       still auf None, und jede Welt sah aus wie eine Schwelle."""
    m=sorted(menge); aus=[]
    if not m: return aus
    for _ in range(max(0,n_ketten)):
        p=list(m); rnd.shuffle(p); kette=[]; gesehen=set()
        for a in sorted(anteile):
            k=int(round(a*len(m)))
            if k<1 or k>=len(m) or k in gesehen: continue
            gesehen.add(k); kette.append((a,k,sorted(p[:k])))
        if kette: aus.append(kette)
    return aus
def teilmenge_zahl(kandidaten,n,rnd):
    """n Paare aus einer Vorgabe ziehen - angeglichen wird die EXPERTENZAHL."""
    k=[q for q in kandidaten]
    if not k or n<=0: return []
    rnd.shuffle(k)
    return sorted(k[:min(n,len(k))])
def kandidaten_kontrolle(zaehler,ausschluss):
    """Woraus die Zufallskontrolle gezogen wird: nur Experten, die im
       Bezugsarm ueberhaupt FEUERN - wer nie an der Reihe ist, laesst sich
       auch nicht sperren -, und nichts aus der gepruefigen Menge."""
    aus=set()
    for m in ausschluss: aus|=set(m)
    return sorted(q for q in zaehler if zaehler[q]>0 and q not in aus)
def senkt(k_bas,k,p,alpha=0.05):
    """Senkt dieser Eingriff, ohne Vergleich mit einer Kontrolle? So und nur
       so wird die Kontrolle selbst beurteilt - sie hat keine eigene."""
    return "TRAEGT" if (k<k_bas and p<alpha) else "STILL"
def anteil_wirkung(k_bas,k_teil,k_voll):
    """Welcher Bruchteil der Wirkung der VOLLEN Menge? Ohne Nenner - also
       wenn die volle Menge selbst nichts tut - gibt es keine Kurve, und die
       Funktion sagt das mit None statt mit einer Zahl."""
    n=k_bas-k_voll
    if n<=0: return None
    return (k_bas-k_teil)/float(n)
def median(xs):
    v=sorted(x for x in xs if x is not None)
    if not v: return None
    m=len(v)//2
    return v[m] if len(v)%2 else 0.5*(v[m-1]+v[m])
def halbwert(punkte,schwelle=0.5):
    """Kleinster Anteil der Menge, bei dem die halbe Wirkung erreicht ist.

       Die volle Menge hat definitionsgemaess die Wirkung 1.0, also gibt es
       immer eine Antwort; im schlechtesten Fall ist sie 1.0."""
    for a,w in sorted(punkte):
        if w is not None and w>=schwelle: return a
    return 1.0
def form_der_kurve(h,unten=0.4,oben=0.8):
    """Eine lineare Kurve haette den Halbwert bei 0.5. Deutlich darunter heisst
       'ein Teil traegt mehr', deutlich darueber 'es braucht fast alles'."""
    if h<=unten: return "WENIGE-TRAGEN"
    if h>=oben: return "SCHWELLE"
    return "ETWA-LINEAR"
def traegt_gruppe(je_arm,arme):
    u=[je_arm.get(a) for a in arme if a in je_arm]
    if not u: return False
    return sum(1 for x in u if x in TRAEGT_ALLE)>=max(2,(len(u)+1)//2)
def urteil_kurve(formen,je_voll,je_zufall,arme=KONSTRUIERT):
    """Zwei Sperren, dann erst die Form.

       Die Kontrolle kommt ZUERST. Senkt eine beliebige gleich grosse Menge
       die Arme ebenso, misst der Lauf die Eingriffsgroesse statt der Menge -
       dann ist nicht nur die Kurve wertlos, sondern auch das Urteil ueber die
       Positivkontrolle: die wirkt in so einer Welt zwar, laesst sich aber
       nicht mehr von der Kontrolle unterscheiden und heisst dann
       NUR-STOERUNG. Stuende die Positivkontrolle vorn, meldete der Lauf einen
       Messkettenfehler, wo in Wahrheit die Dosis zu gross ist.

       Erst danach der Nenner: ohne tragende Positivkontrolle gibt es keine
       Kurve."""
    if traegt_gruppe(je_zufall,arme): return "KONTROLLE-STOERT"
    if not traegt_gruppe(je_voll,arme): return "POSITIVKONTROLLE-FEHLT"
    f=[formen[a] for a in arme if a in formen and formen[a]]
    if not f: return "KEINE-ARME"
    for name in ("SCHWELLE","ETWA-LINEAR","WENIGE-TRAGEN"):
        if sum(1 for x in f if x==name)>=max(2,(len(f)+1)//2): return name
    return "UNEINHEITLICH"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_BSP=int(globals().get("N_BSP",48)); MAX_NEW=int(globals().get("MAX_NEW",96))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260814))
MIN_BSP=int(globals().get("MIN_BSP",20))
N_ABL=int(globals().get("N_ABL",48))
N_KETTEN=int(globals().get("N_KETTEN",3))
N_PRUEF=int(globals().get("N_PRUEF",8))
WIEDERHOLUNG=int(globals().get("WIEDERHOLUNG",0))
def saat(zweck,schl):
    h=2166136261
    for c in (zweck+"/"+schl).encode():
        h=((h^c)*16777619)&0xFFFFFFFF
    return (SEED+1000003*WIEDERHOLUNG+h)%(2**31-1)
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
ROH_PROMPT=PROMPTS[ZIEL_ID]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
EXPM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; INTER=int(e0.intermediate_dim); NEXP=int(GU.shape[0])
    TOPK=int(cfg.num_experts_per_tok)
    ARCH_OK=(GU.ndim==3 and GU.shape[1]==2*INTER and GU.shape[2]==cfg.hidden_size)
    print("  %d Schichten | %d Experten je Schicht | top-%d | %d Paare gesamt"
          %(len(EXPM),NEXP,TOPK,len(EXPM)*NEXP))
    print("  Formen wie erwartet: %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    KURVE_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- Werkzeuge ---------------------------------------------------
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
class Maske:
    """Setzt den Router-Anteil ganzer Experten auf null. Reiner Vorwaerts-Haken:
       kein Sicherungsabzug, wirkt an JEDER Position, restlos umkehrbar."""
    def __init__(self,verboten):
        self.verboten={l:set(v) for l,v in verboten.items() if v}
        self.griffe=[]; self.pruefen=False
        self.getroffen=0; self.rest=0.0; self.formen=None
    def _mach(self,bad):
        def h(mod,args):
            idx=args[1]; w=args[2]
            if idx.shape!=w.shape:
                self.formen=(tuple(idx.shape),tuple(w.shape)); return None
            tr=torch.isin(idx,bad)
            neu=w.masked_fill(tr,0.0)
            if self.pruefen:
                self.getroffen+=int(tr.sum().item())
                if bool(tr.any()):
                    self.rest=max(self.rest,float(neu[tr].abs().max().item()))
            return (args[0],idx,neu)+tuple(args[3:])
        return h
    def __enter__(self):
        for l,vs in self.verboten.items():
            W=EXPM[l].gate_up_proj
            bad=torch.tensor(sorted(vs),device=W.device,dtype=torch.long)
            self.griffe.append(EXPM[l].register_forward_pre_hook(self._mach(bad)))
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
def nur_logits(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        return model(ids).logits[0,-1].float().cpu().numpy()
def mit_maske(verboten,text,n,startwert,pruef_texte,pruef_logits):
    """Die gesperrten Plaetze werden ueber DIESELBEN Texte gezaehlt, auf denen
       die Dosis angeglichen wurde - Prompt UND Antwort.

       Zuerst stand hier ein einzelner Prompt, waehrend die Angleichung ueber
       Prompt+Antwort lief. Geplant und gemessen wichen dadurch um mehr als
       das Doppelte voneinander ab (33 gegen 17 Plaetze), ohne dass eine der
       beiden Zahlen falsch ausgesehen haette."""
    with Maske(verboten) as M:
        M.pruefen=True
        lg=[nur_logits(t) for t in pruef_texte]
        M.pruefen=False
        if M.formen is not None:
            raise RuntimeError("Router-Formen passen nicht: idx %s, gewichte %s"%M.formen)
        wirk=max(float(np.abs(a-b).max()) for a,b in zip(lg,pruef_logits))
        aus=zieh(text,n,startwert)
    return aus,M.getroffen,M.rest,wirk
def plaetze_ganz(texte):
    z=collections.Counter()
    for text in texte:
        ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
        fang={}
        def mach(l):
            def h(mod,args): fang[l]=args[1].detach(); return None
            return h
        hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
        try:
            with torch.no_grad(): model(ids)
        finally:
            for h in hs: h.remove()
        for l,idx in fang.items():
            for e in idx.reshape(-1).tolist(): z[(l,int(e))]+=1
    return z
def hole_routing(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear()
            model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
    finally:
        for h in hs: h.remove()
    r=set()
    for l,idx in fang.items():
        for e in idx.reshape(-1,idx.shape[-1])[-1].tolist(): r.add((l,int(e)))
    return sorted(r)
# ---------------- 1  Die geprueffte Menge -----------------------------------
print(""); print("="*80); print("1  DIE MENGE AUS PHASE 12")
print("="*80)
ARMTEXT={}
for schl,phrase,_ in ARME:
    t2,ok=setze_arm(ROH_PROMPT,phrase)
    assert ok,"Phrase %r nicht im Prompt gefunden"%PHRASE
    ARMTEXT[schl]=prompt_text(t2)
MASS={"NEU":"englisch","JA":"kana","SR":"kyrillisch","RU":"kyrillisch",
      "BR1":"braille","MORSE":"morse"}
MENGE=sorted(set(hole_routing(ARMTEXT["JA"]))-set(hole_routing(ARMTEXT[BEZUG])))
print("  Differenz an der Entscheidungsstelle, Japanisch gegen Englisch:")
print("  %d Paare in %d Schichten: %s"
      %(len(MENGE),len(set(l for l,_ in MENGE)),sorted(set(l for l,_ in MENGE))))
print("  (selbst hergeleitet, nicht eingetragen - so bleibt ein Nachlauf in")
print("   sich stimmig und keine Zahl haengt an einer Uebertragung)")
if len(MENGE)<8:
    KURVE_RESULTS=dict(verdict="MENGE-ZU-KLEIN",arch_ok=True,
                       menge=[list(q) for q in MENGE])
    print(""); print("VERDIKT: MENGE-ZU-KLEIN"); wc_save_all(); raise SystemExit(0)
# ---------------- 2  Grundrate und Prueftexte -------------------------------
print(""); print("="*80)
print("2  GRUNDRATE JE ARM (%d Beispiele, gesampelt)"%N_BSP)
print("="*80)
t0=time.time(); ANTW={}; GRUND={}
for schl,phrase,zweck in ARME:
    roh=zieh(ARMTEXT[schl],N_BSP,saat("grund",schl))
    f=zielmass(MASS[schl])
    ANTW[schl]=[t for t in roh if f(t)]
    GRUND[schl]=len(ANTW[schl])
    print("  %-6s %2d/%-3d treffen das Zielmass  %s"%(schl,GRUND[schl],N_BSP,zweck))
print("  (%.0f s)"%(time.time()-t0))
PRUEF={schl:([ARMTEXT[schl]+a for a in ANTW[schl][:N_PRUEF]] or [ARMTEXT[schl]])
       for schl,_,_ in ARME}
PLAETZE={schl:plaetze_ganz(PRUEF[schl]) for schl,_,_ in ARME}
LEBEN=[s for s in KONSTRUIERT if GRUND[s]>=MIN_BSP]
SPEZ=[s for s in LEXIKALISCH if GRUND[s]>=MIN_BSP]
print("  Kurve auf: %s | Spezifitaetspruefung auf: %s"
      %(", ".join(LEBEN) or "-",", ".join(SPEZ) or "-"))
if len(LEBEN)<2:
    KURVE_RESULTS=dict(verdict="ZU-WENIGE-ARME",arch_ok=True,grund=GRUND,
                       menge=[list(q) for q in MENGE])
    print(""); print("VERDIKT: ZU-WENIGE-ARME")
    print("  Unter zwei konstruierenden Armen mit ausreichender Grundrate ist")
    print("  keine Mehrheit zu bilden - und eine Kurve aus einem Arm ist keine.")
    wc_save_all(); raise SystemExit(0)
# ---------------- 3  Die Ketten ---------------------------------------------
print(""); print("="*80); print("3  DREI GESCHACHTELTE KETTEN")
print("="*80)
rnd=random.Random(saat("ketten","alle"))
KETTEN=ketten(MENGE,N_KETTEN,ANTEILE,rnd)
_kand=kandidaten_kontrolle(PLAETZE[BEZUG],[MENGE])
# Am TOPF geprueft, nicht an der Ziehung: waere die Menge nicht ausgeschlossen,
# traefe eine einzelne Ziehung sie nur mit etwa halber Wahrscheinlichkeit - und
# ein Lauf, in dem die Kontrolle sich selbst enthaelt, saehe dann in der
# Haelfte der Faelle sauber aus.
assert not (set(_kand)&set(MENGE)),"Kandidatentopf enthaelt die gepruefte Menge"
ZUF=teilmenge_zahl(_kand,len(MENGE),rnd)
ZAHL_OK=(len(ZUF)==len(MENGE))
assert not (set(ZUF)&set(MENGE)),"Kontrolle enthaelt die gepruefte Menge"
print("  je Kette eine Permutation, die Stufen sind INEINANDER enthalten:")
for r,kette in enumerate(KETTEN):
    print("    Kette %d: %s"%(r+1,", ".join("%d Paare (soll %.0f %%, ist %.0f %%)"
                                            %(k,100*a,100.0*k/len(MENGE))
                                            for a,k,_ in kette)))
for r,kette in enumerate(KETTEN):
    for i in range(len(kette)-1):
        assert set(kette[i][2])<set(kette[i+1][2]),"Kette %d nicht geschachtelt"%(r+1)
STUFEN=[]
for r,kette in enumerate(KETTEN):
    for a,k,teil in kette:
        # Schluessel ist der SOLLANTEIL. k/len(MENGE) weicht davon ab, sobald
        # die Menge nicht durch fuenf teilbar ist, und die Kurve fiele
        # auseinander, ohne dass eine Zahl falsch aussaehe.
        STUFEN.append(dict(name="k%02d/%d"%(k,r+1),menge=teil,anteil=a,
                           ist_anteil=k/float(len(MENGE)),kette=r+1,k=k))
STUFEN.append(dict(name="VOLL",menge=list(MENGE),anteil=1.0,ist_anteil=1.0,
                   kette=0,k=len(MENGE)))
STUFEN.append(dict(name="ZUFALL",menge=ZUF,anteil=None,ist_anteil=None,
                   kette=0,k=len(ZUF)))
print("  dazu VOLL (%d, Positivkontrolle und Nenner) und ZUFALL (%d von aussen)"
      %(len(MENGE),len(ZUF)))
print("  gleiche Zahl bei der Kontrolle: %s"%("ja" if ZAHL_OK else "NEIN"))
print("  %d Bedingungen je Kurvenarm, %d Ziehungen je Bedingung"%(len(STUFEN),N_ABL))
if not ZAHL_OK:
    KURVE_RESULTS=dict(verdict="ZAHL-NICHT-ANGEGLICHEN",arch_ok=True,zahl_ok=False,
                       menge=[list(q) for q in MENGE])
    print(""); print("VERDIKT: ZAHL-NICHT-ANGEGLICHEN")
    print("  Die Zufallskontrolle laesst sich nicht auf %d Experten bringen."%len(MENGE))
    print("  Ohne sie lizenziert nichts die Kurve - jeder Abfall koennte")
    print("  blosse Eingriffsgroesse sein.")
    wc_save_all(); raise SystemExit(0)
print("")
print("  gesperrte Router-Plaetze je Bedingung (ueber Prompt+Antwort):")
print("  %-10s"%"Bedingung"+"".join("%9s"%s for s in LEBEN+SPEZ))
GEPLANT={}
for st in STUFEN:
    GEPLANT[st["name"]]={s:sum(PLAETZE[s][q] for q in st["menge"]) for s in LEBEN+SPEZ}
    print("  %-10s"%st["name"]
          +"".join("%9d"%GEPLANT[st["name"]][s] for s in LEBEN+SPEZ))
# ---------------- 4  Ablation -----------------------------------------------
print(""); print("="*80); print("4  ABLATION")
print("="*80)
L_REF=[nur_logits(t) for t in PRUEF[BEZUG]]
ERG={}; t0=time.time()
for schl in LEBEN+SPEZ:
    f=zielmass(MASS[schl]); L0=[nur_logits(t) for t in PRUEF[schl]]
    noetig=STUFEN if schl in LEBEN else [st for st in STUFEN
                                         if st["name"] in ("VOLL","ZUFALL")]
    a0=zieh(ARMTEXT[schl],N_ABL,saat("basis",schl))
    k0=sum(1 for t in a0 if f(t))
    zeile=dict(mass=MASS[schl],k_basis=k0,n=N_ABL,stufen={})
    for st in noetig:
        a,g,r,w=mit_maske(nach_schicht(st["menge"]),ARMTEXT[schl],N_ABL,
                          saat(st["name"],schl),PRUEF[schl],L0)
        assert r<1e-6,"maskierter Experte behaelt Router-Gewicht"
        assert g>0,"kein einziger Router-Platz gesperrt (%s/%s)"%(st["name"],schl)
        k=sum(1 for t in a if f(t))
        zeile["stufen"][st["name"]]=dict(k=k,plaetze=g,anteil=st["anteil"],
                                         ist_anteil=st["ist_anteil"],
                                         kette=st["kette"],groesse=st["k"],logit=w)
    zu=zeile["stufen"]["ZUFALL"]
    p_zu=fisher2x2(zu["k"],N_ABL-zu["k"],k0,N_ABL-k0)
    for nm,d in zeile["stufen"].items():
        d["p"]=fisher2x2(d["k"],N_ABL-d["k"],k0,N_ABL-k0)
        d["je_platz"]=wirkung_je_platz(k0,d["k"],N_ABL,d["plaetze"])
        d["urteil"]=urteil_arm_dosis(k0,N_ABL,d["k"],d["p"],zu["k"],p_zu,
                                     d["plaetze"],zu["plaetze"])
        d["wirkung"]=None
    zeile["stufen"]["ZUFALL"]["urteil"]="KONTROLLE"
    v=zeile["stufen"]["VOLL"]
    for nm,d in zeile["stufen"].items():
        if nm!="ZUFALL": d["wirkung"]=anteil_wirkung(k0,d["k"],v["k"])
    ERG[schl]=zeile
    print("  %-6s Grundrate %2d/%d = %3.0f %%   (%s)"
          %(schl,k0,N_ABL,100.0*k0/N_ABL,"Kurve" if schl in LEBEN else "nur Spezifitaet"))
    print("    %-10s %6s %7s %9s %9s  %s"
          %("Bedingung","Zahl","Treffer","p","je Platz","Urteil"))
    for st in noetig:
        d=zeile["stufen"][st["name"]]
        print("    %-10s %6d %3d %3.0f%% %9.4f %9.2f  %s%s"
              %(st["name"],d["groesse"],d["k"],100.0*d["k"]/N_ABL,d["p"],
                d["je_platz"],d["urteil"],
                "" if d["wirkung"] is None else "  Wirkung %.2f"%d["wirkung"]))
print("  (%.0f s)"%(time.time()-t0))
ABW=max(float(np.abs(nur_logits(t)-r).max()) for t,r in zip(PRUEF[BEZUG],L_REF))
print("  WIEDERHERSTELLUNG: alle Haken abgenommen (Abweichung %.2e)"%ABW)
assert ABW<1e-2,"Haken haengen noch"
# ---------------- 5  Die Kurve ----------------------------------------------
print(""); print("="*80); print("5  DIE KURVE")
print("="*80)
JE_VOLL={s:ERG[s]["stufen"]["VOLL"]["urteil"] for s in ERG}
# Die Zufallskontrolle hat keine eigene Kontrolle - sie wird nackt gegen die
# Grundrate gemessen. Ein Eingriff, der so schon signifikant senkt, macht die
# ganze Kurve unlesbar, und die Sperre dafuer steht im Urteil.
JE_ZUFALL={s:senkt(ERG[s]["k_basis"],ERG[s]["stufen"]["ZUFALL"]["k"],
                   ERG[s]["stufen"]["ZUFALL"]["p"]) for s in ERG}
KURVE={}; FORMEN={}; HALB={}
for s in LEBEN:
    punkte=[]
    for a in sorted(ANTEILE):
        w=median([d["wirkung"] for d in ERG[s]["stufen"].values()
                  if d["anteil"] is not None and abs(d["anteil"]-a)<1e-9])
        punkte.append((a,w))
    punkte.append((1.0,1.0 if ERG[s]["stufen"]["VOLL"]["wirkung"] is not None else None))
    KURVE[s]=punkte
    # Ohne tragende Positivkontrolle hat dieser Arm keinen Nenner. Dann gibt es
    # fuer ihn auch keine Form - und er darf im Mehrheitsurteil nicht
    # mitstimmen, statt stillschweigend als SCHWELLE zu zaehlen.
    if ERG[s]["stufen"]["VOLL"]["wirkung"] is None:
        HALB[s]=None; FORMEN[s]=None
    else:
        HALB[s]=halbwert(punkte); FORMEN[s]=form_der_kurve(HALB[s])
print("  Median ueber die %d Ketten, als Bruchteil der Wirkung der vollen Menge."%N_KETTEN)
print("  Eine gerade Linie haette bei 20/40/60/80 %% die Werte .20/.40/.60/.80.")
print("")
_gr={a:k for a,k,_ in (KETTEN[0] if KETTEN else [])}
print("  %-6s"%"Arm"+"".join("%9s"%("%.0f%%/%d"%(100*a,_gr.get(a,0)))
                             for a in sorted(ANTEILE))
      +"%9s %10s  %s"%("100%%/%d"%len(MENGE),"Halbwert","Form"))
for s in LEBEN:
    print("  %-6s"%s
          +"".join("%9s"%("-" if w is None else "%.2f"%w) for a,w in KURVE[s][:-1])
          +"%9s %9s  %s"%("-" if HALB[s] is None else "1.00",
                          "-" if HALB[s] is None else "%.0f %%"%(100.0*HALB[s]),
                          FORMEN[s] or "kein Nenner"))
print("")
print("  Streuung der Ketten (kleinster und groesster Wert je Stufe):")
for s in LEBEN:
    aus=[]
    for a in sorted(ANTEILE):
        v=[d["wirkung"] for d in ERG[s]["stufen"].values()
           if d["anteil"] is not None and abs(d["anteil"]-a)<1e-9 and d["wirkung"] is not None]
        aus.append("-" if not v else "%.2f..%.2f"%(min(v),max(v)))
    print("    %-6s %s"%(s," | ".join(aus)))
print("")
print("  Dieselbe Wirkung gegen die gesperrten PLAETZE - faellt die Kurve hier")
print("  zusammen, war nie die Zahl der Experten die wirksame Groesse:")
for s in LEBEN:
    ps=sorted((d["plaetze"],d["wirkung"]) for d in ERG[s]["stufen"].values()
              if d["anteil"] is not None and d["wirkung"] is not None)
    print("    %-6s %s"%(s," ".join("%d:%.2f"%(p,w) for p,w in ps)))
CODE=urteil_kurve(FORMEN,JE_VOLL,JE_ZUFALL,tuple(LEBEN))
print(""); print("="*80); print("VERDIKT: %s"%CODE); print("="*80)
if CODE=="POSITIVKONTROLLE-FEHLT":
    print("  Die vollen %d senken die konstruierenden Arme in DIESEM Lauf"%len(MENGE))
    print("  nicht - obwohl sie es in Phase 12 zweimal und in Phase 15 noch")
    print("  einmal getan haben. Ohne Nenner gibt es keine Kurve, und alles")
    print("  darunter waere unlesbar.")
elif CODE=="KONTROLLE-STOERT":
    print("  Auch %d BELIEBIGE Experten senken die Arme. Dann misst der Lauf"%len(ZUF))
    print("  die Eingriffsgroesse und nicht die Menge, und die Kurve sagt")
    print("  nichts ueber die 42.")
elif CODE=="SCHWELLE":
    print("  Die halbe Wirkung kommt erst, wenn ueber drei Viertel der Menge")
    print("  gesperrt sind. Die Menge ist damit REDUNDANT: es gibt mehrere")
    print("  Wege, die einander vertreten, und einzelne Experten oder kleine")
    print("  Teilmengen sind entbehrlich. Das erklaert auch, warum Phase 15")
    print("  mit einem Achtel nichts gefunden hat - dort war nichts zu finden.")
elif CODE=="ETWA-LINEAR":
    print("  Die Wirkung waechst ungefaehr mit der Zahl der gesperrten")
    print("  Experten. Die Menge ist dann eine SUMME und kein Schaltkreis:")
    print("  jeder Experte traegt seinen Anteil, keiner ist ausgezeichnet.")
elif CODE=="WENIGE-TRAGEN":
    print("  Schon ein kleiner Teil der Menge bringt die halbe Wirkung. Dann")
    print("  gibt es doch einen Kern innerhalb der %d, und Phase 15 hat ihn"%len(MENGE))
    print("  mit EINER Ziehung nur verfehlt. Die naechste Frage waere, welche")
    print("  Experten das sind - dafuer braucht es gezielte Ziehungen statt")
    print("  zufaelliger.")
else:
    print("  Die Arme zeigen verschiedene Formen, und keine hat die Mehrheit.")
    print("  Dann gibt es keine gemeinsame Kurve - moeglich ist, dass die drei")
    print("  Arme verschieden auf dieselbe Menge zugreifen.")
print("")
print("  Halbwert je Arm: %s"
      %", ".join("%s %s (%s)"%(s,"-" if HALB[s] is None else "%.0f %%"%(100.0*HALB[s]),
                               FORMEN[s] or "kein Nenner") for s in LEBEN))
print("  volle Menge    : %s"%", ".join("%s %s"%(s,JE_VOLL[s]) for s in sorted(JE_VOLL)))
print("  Zufallsmenge   : %s"%", ".join("%s %s"%(s,JE_ZUFALL[s]) for s in sorted(JE_ZUFALL)))
if SPEZ:
    print("")
    print("  Spezifitaet - die kyrillischen Arme duerfen nicht fallen:")
    for s in SPEZ:
        e=ERG[s]; v=e["stufen"]["VOLL"]; z=e["stufen"]["ZUFALL"]
        print("    %-6s ohne %2d/%d | voll %2d (p=%.4f) | zufall %2d (p=%.4f)"
              %(s,e["k_basis"],N_ABL,v["k"],v["p"],z["k"],z["p"]))
print("")
print("  Vorbehalt: die Stufen sind auf die ZAHL der Experten gelegt, nicht auf")
print("  die Plaetze - anders ist es nicht zu machen, 8 Experten sperren nun")
print("  einmal weniger als 42. Die Platzzeile oben zeigt, wie stark Zahl und")
print("  Dosis auseinanderlaufen. Und %d Ketten geben eine Streuung, keine"%N_KETTEN)
print("  Verteilung: ein Halbwert dicht an einer Grenze ist nicht belastbar.")
KURVE_RESULTS=dict(verdict=CODE,arch_ok=True,zahl_ok=bool(ZAHL_OK),
    prompt_id=ZIEL_ID,n_bsp=N_BSP,n_abl=N_ABL,n_ketten=N_KETTEN,
    wiederholung=WIEDERHOLUNG,anteile=list(ANTEILE),
    menge=[list(q) for q in MENGE],zufall=[list(q) for q in ZUF],
    ketten=[[[a,k,[list(q) for q in teil]] for a,k,teil in kette] for kette in KETTEN],
    grund=GRUND,ergebnis=ERG,
    plaetze_geplant={nm:GEPLANT[nm] for nm in GEPLANT},kurve={s:[[a,w] for a,w in KURVE[s]] for s in KURVE},
    halbwert=HALB,formen=FORMEN,je_voll=JE_VOLL,je_zufall=JE_ZUFALL,
    leben=list(LEBEN),spezifitaet=list(SPEZ),abweichung_ende=ABW)
wc_save("kurve_stufen",{s:{nm:ERG[s]["stufen"][nm]["k"] for nm in ERG[s]["stufen"]}
                        for s in ERG})
wc_save_all()
